# Tema 2: Estructura de proyectos en Python

En este cuaderno vas a aprender como organizar un proyecto Python de forma clara, mantenible y profesional.

## Objetivos

Al finalizar este tema podras:
- Reconocer los ficheros tipicos de un proyecto Python.
- Entender para que sirve cada fichero.
- Montar una estructura basica minima para empezar proyectos reales.
- Manejar rutas de ficheros en diferentes situaciones.
- Entender el papel de pyproject.toml y su uso con uv.

## 1) Por que importa la estructura

Una buena estructura de proyecto te ayuda a:
- Encontrar codigo rapido.
- Separar logica de negocio, pruebas y configuracion.
- Escalar el proyecto sin caos.
- Facilitar trabajo en equipo.
- Reducir errores por rutas o imports mal organizados.

## 2) Estructura basica recomendada

Ejemplo de estructura minima para un proyecto Python:

```text
mi_proyecto/
  pyproject.toml
  README.md
  .gitignore
  .venv/
  src/
    mi_proyecto/
      __init__.py
      main.py
      utils.py
  tests/
    test_main.py
```

Nota: la carpeta .venv no se sube al repositorio.

## 3) Tipos de ficheros y para que sirven

### pyproject.toml
Fichero central moderno para configurar el proyecto:
- Metadatos del paquete (nombre, version, descripcion).
- Dependencias del proyecto.
- Configuracion de herramientas (ruff, pytest, etc.).
- Sistema de build cuando empaquetas.

### README.md
Explica que hace el proyecto, como instalarlo y como usarlo.

### .gitignore
Lista ficheros/carpetas que git no debe versionar (ejemplo: .venv, caches, builds).

### src/
Contiene el codigo fuente real del proyecto.

### tests/
Contiene los tests automaticos.

### __init__.py
Marca la carpeta como paquete Python y permite organizar imports.

### requirements.txt (opcional segun flujo)
Listado congelado de dependencias para reproducir entorno en sistemas tradicionales.

## 4) pyproject.toml: estructura minima y uso

Ejemplo basico:

```toml
[project]
name = "mi-proyecto"
version = "0.1.0"
description = "Proyecto de ejemplo"
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
  "flask==3.0.0",
  "pytest==8.2.1"
]

[tool.pytest.ini_options]
pythonpath = ["src"]
testpaths = ["tests"]
```

Con esto ya tienes una base funcional para empezar un proyecto y ejecutar pruebas.

## 5) Flujo con uv y pyproject.toml

Flujo habitual con uv:

```bash
mkdir mi_proyecto
cd mi_proyecto
uv init
uv venv

# Linux y macos
source .venv/bin/activate
# Windows (CMD)
.venv\Scripts\activate
# Windows (PowerShell)
.venv\Scripts\Activate.ps1

uv add flask pytest
uv run pytest
```

Que hace cada paso:
- uv init: crea estructura inicial y pyproject.toml.
- uv venv: crea entorno virtual.
- uv add: agrega dependencias y actualiza pyproject.toml.
- uv run pytest: ejecuta tests dentro del entorno gestionado.

## 6) Flujo equivalente sin uv (venv + pip)

```bash
mkdir mi_proyecto
cd mi_proyecto
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install flask pytest
pip freeze > requirements.txt
pytest
```

En este flujo, pyproject.toml puede existir igualmente para configuracion de herramientas,
pero la gestion principal de dependencias suele quedar en requirements.txt.

## 7) Como organizar el codigo dentro de src

No todos los proyectos se organizan igual. Estas plantillas te ayudan a decidir.

### Caso A: proyecto pequeno (script mejorado)
Usalo cuando el proyecto todavia es simple, pero quieres orden minimo:

```text
src/
  mi_proyecto/
    __init__.py
    main.py
    utils.py
```

### Caso B: libreria reutilizable
Usalo cuando tu codigo se importa desde otros proyectos:

```text
src/
  mi_libreria/
    __init__.py
    api.py
    validators.py
    exceptions.py
```

### Caso C: API web (FastAPI o Flask con capas)
Usalo cuando tienes endpoints, logica y acceso a datos separados:

```text
src/
  app/
    __init__.py
    main.py
    routers/
      __init__.py
      users.py
    services/
      __init__.py
      user_service.py
    repositories/
      __init__.py
      user_repo.py
    models/
      __init__.py
      user.py
```

### Caso D: aplicacion mas grande por dominios
Usalo cuando el proyecto crece y necesitas modularidad por funcionalidad:

```text
src/
  app/
    __init__.py
    common/
      config.py
      db.py
    alumnos/
      controller.py
      service.py
      repository.py
      schemas.py
    matriculas/
      controller.py
      service.py
      repository.py
      schemas.py
```

Regla practica:
- Si un archivo supera tamano o mezcla responsabilidades, separalo.
- Nombres de carpetas por responsabilidad, no por tecnologia suelta.
- Evita meter toda la logica en main.py.

## 8) Casos concretos de rutas de ficheros

Trabajar con rutas bien es clave para evitar errores al mover proyectos entre equipos o sistemas operativos.

In [ ]:
from pathlib import Path

# Caso 1: ruta relativa dentro del proyecto
raiz = Path.cwd()
ruta_config = raiz / 'config' / 'settings.toml'
print('Ruta config:', ruta_config)

# Caso 2: construir ruta al archivo de datos en data/
ruta_datos = raiz / 'data' / 'alumnos.csv'
print('Ruta datos:', ruta_datos)

# Caso 3: obtener ruta absoluta normalizada
print('Absoluta datos:', ruta_datos.resolve())

### Caso extra A: cargar un fichero respecto al propio script

Cuando ejecutas scripts desde distintas carpetas, usar Path(__file__) evita errores:

```python
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
RUTA_PLANTILLA = BASE_DIR / 'templates' / 'index.html'
```

### Caso extra B: rutas en tests

En tests, es comun necesitar rutas al directorio raiz del proyecto:

```python
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
FIXTURE = ROOT / 'tests' / 'fixtures' / 'usuarios.json'
```

Este patron evita depender del directorio actual desde donde lances pytest.

### Caso extra C: separar rutas de usuario y rutas internas

Si el usuario pasa una ruta por entrada o argumento:
- Validala.
- Resuelvela a absoluta si es necesario.
- No mezcles directamente con rutas internas sin comprobarla.

In [ ]:
from pathlib import Path

def preparar_ruta_usuario(ruta: str) -> Path:
    p = Path(ruta).expanduser()
    return p.resolve()

print(preparar_ruta_usuario('~/documentos/mi_fichero.txt'))

## 9) Checklist minimo de proyecto

Antes de empezar a desarrollar, verifica:
- Existe pyproject.toml.
- Tienes entorno virtual creado (.venv).
- Estructura src/ y tests/ creada.
- README.md inicial con instrucciones basicas.
- .gitignore con .venv y caches.
- Dependencias instaladas y registradas.

## 10) Resumen

Una estructura clara y el uso correcto de pyproject.toml te permiten construir proyectos Python mas ordenados, reproducibles y faciles de mantener.

Con uv, ademas, gestionas entorno y dependencias de forma moderna y muy rapida.